# Final Project Phase 3 Summary
This Jupyter Notebook (.ipynb) will serve as the skeleton file for your submission for Phase 3 of the Final Project. Complete all sections below as specified in the instructions for the project, covering all necessary details. We will use this to grade your individual code (Do this whether you are in a group or not). Good luck! <br><br>

Note: To edit a Markdown cell, double-click on its text.

## Jupyter Notebook Quick Tips
Here are some quick formatting tips to get you started with Jupyter Notebooks. This is by no means exhaustive, and there are plenty of articles to highlight other things that can be done. We recommend using HTML syntax for Markdown but there is also Markdown syntax that is more streamlined and might be preferable. 
<a href = "https://towardsdatascience.com/markdown-cells-jupyter-notebook-d3bea8416671">Here's an article</a> that goes into more detail. (Double-click on cell to see syntax)

# Heading 1
## Heading 2
### Heading 3
#### Heading 4
<br>
<b>BoldText</b> or <i>ItalicText</i>
<br> <br>
Math Formulas: $x^2 + y^2 = 1$
<br> <br>
Line Breaks are done using br enclosed in < >.
<br><br>
Hyperlinks are done with: <a> https://www.google.com </a> or 
<a href="http://www.google.com">Google</a><br>

# Video Presentation

If you uploaded your Video Presentation to Bluejeans, YouTube, or any other streaming services, please provide the link here:


*   Video Presentation Link
https://youtu.be/ReZg--SswjA

Make sure the video sharing permissions are accessible for anyone with the provided link.

# Data Collection and Cleaning


Transfer/update the data collection and cleaning you created for Phase II below. You may include additional cleaning functions if you have extra datasets. If no changes are necessary, simply copy and paste your phase II parsing/cleaning functions.


## Downloaded Dataset Requirement



In [1]:
import pandas as pd
import numpy as np
import re

def data_parser():

  df = pd.read_csv("pbp-2024.csv")
  df.drop(["Unnamed: 10", "Unnamed: 12", "Unnamed: 16", "Unnamed: 17"], axis=1, inplace = True)

  df.sort_values(by=["GameId", "Quarter", "Minute", "Second"], ascending=[True, True, False, False], inplace=True)

  for i in range(1, len(df)):
      currentIndex = df.index[i]
      previousIndex = df.index[i - 1]
      
      if df.loc[currentIndex, "YardLine"] == 0:
          df.loc[currentIndex, "YardLine"] = df.loc[previousIndex, "YardLine"]

  df["PassType"] = df["PassType"].fillna("NOT A PASS")
  df["Challenger"] = df["Challenger"].fillna("NO CHALLENGE")
  df = df.drop("RushDirection", axis = 1)
  df["PenaltyTeam"] = df["PenaltyTeam"].fillna("NO PENALTY")
  df["PenaltyType"] = df["PenaltyType"].fillna("NO PENALTY")
  df["Formation"] = df["Formation"].fillna("NO FORMATION")
  df["PlayType"] = df["PlayType"].fillna("NO PLAY")

  df["IsScramble"] = np.where(df["PlayType"].str.contains("SCRAMBLE", regex = False), 1, 0)
  df["IsRush"] = df["PlayType"].apply(lambda x: 0 if "SCRAMBLE" in x else 1)

  df = df[df["GameDate"] < "2025-01-11"]
    
  df.to_csv("new-pbp.csv", index=False)

############ Function Call ############
data_parser()

## Web Collection Requirement \#1


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def web_parser1():
    url="https://www.spotrac.com/nfl/rankings/player/_/year/2024/position/qb/sort/contract_average"

    response = requests.get(url, headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/118.0.5993.117 Safari/537.36"
    })

    soup = BeautifulSoup(response.text, "html.parser")

    table = soup.find("ul", {"class": "list-group mb-4 not-premium"})

    rows = table.find_all("li")

    data = []

    for row in rows:
        myDict = {}
        if len(row["class"]) < 2:
            continue
        name = row.find_all("a")[0].text.strip()
        salary = row.find_all("span", {"class": "medium"})[0].text.strip()
        myDict[name] = int(salary.strip("$").replace(",", ""))
        data.append(myDict)

    df = pd.DataFrame([(list(d.keys())[0], list(d.values())[0]) for d in data], columns=["Player", "Salary"])
    df.to_csv("salaries.csv", index=False)

############ Function Call ############
web_parser1()

## Web Collection Requirement #2

In [3]:
import requests
import json

def web_parser2():

  players_dict = {"Lamar Jackson": 3916387, "Josh Allen": 3918298, "Joe Burrow": 3915511, "Jared Goff": 3046779, "Jayden Daniels": 4426348, "Baker Mayfield": 3052587, "Patrick Mahomes": 3139477, "Matthew Stafford": 12483, "Jalen Hurts": 4040715, "Jordan Love": 4036378, "Justin Herbert": 4038941, "Geno Smith": 15864, "Sam Darnold": 3912547, "Brock Purdy": 4361741, "C.J. Stroud": 4432577, "Bryce Young": 4685720, "Kyler Murray": 3917315, "Tua Tagovailoa": 4241479, "Dak Prescott": 2577417, "Bo Nix": 4426338, "Drake Maye": 4431452, "Aaron Rodgers": 8439, "Russell Wilson": 14881, "Derek Carr": 16757, "Kirk Cousins": 14880, "Trevor Lawrence": 4360310, "Justin Fields": 4362887, "Caleb Williams": 4431611, "Aidan O'Connell": 4260394, "Michael Penix Jr.": 4360423, "Deshaun Watson":3122840,"Tommy DeVito":4240391,"Carson Wentz":2573079,"Dorian Thomas-Robinson" :4367178,"Skylar Thompson":4036419,"Jake Haener":4243322,"Desmond Ridder":4239086,"Trey Lance":4383351,"Bailey Zappe":4250360,"Brandon Allen":2574511,"Joshua Dobbs":3044720,"Tanner McKee":4685201,"Kenny Pickett":4240703,"Jimmy Garoppolo":16760,"Drew Lock":3924327,"Jacoby Brissett":2578570,"Andy Dalton":14012,"Mac Jones":4241464,"Mason Rudolph":3116407,"Joe Flacco":11252,"Spencer Rattler":4426339,"Cooper Rush":2972515,"Daniel Jones":3917792,"Gardner Minshew":4038524,"Anthony Richardson Sr.":4429084,"Tyler Huntley":4035671,"Will Levis":4361418,"Jameis Winston":2969939,"Malik Willis":4242512}

  stats_dict = {}

  for k, v in players_dict.items():
      player_id = v
      url = f'https://site.web.api.espn.com/apis/common/v3/sports/football/nfl/athletes/{player_id}/gamelog?season=2024'

      response = requests.get(url, headers = {
      "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/118.0.5993.117 Safari/537.36"
      })
      data = response.json()

      for season in data["seasonTypes"]:
          summary = season["summary"]
          if summary["displayName"] == "Regular Season Stats":
              totals = summary["stats"][0]["stats"]

              pyd = int(totals[2].replace(",", ""))   # passing yards
              comp = float(totals[3])  # completion %
              ptd = int(totals[5])   # passing TDs
              inter = int(totals[6]) # interceptions
              ryd = int(totals[12])  # rushing yards
              rtd = int(totals[14])  # rushing TDs
          
              break

      stats_dict[k] = {"Passing Yards": pyd, "Completion Percentage": comp, "Passing Touchdowns": ptd, "Interceptions": inter, "Rushing Yards": ryd, "Rushing Touchdowns": rtd}

  json.dump(stats_dict, open("stats.json", "w"))

############ Function Call ############
web_parser2()

#Inconsistency Revisions
 **If you were requested to revise your inconsistency section from Phase II, enter your responses here. Otherwise, ignore this section.**

For each inconsistency (NaN, null, duplicate values, empty strings, etc.) you discover in your datasets, write at least 2 sentences stating the significance, how you identified it, and how you handled it.

1. 

2. 

3. 

4. (if applicable)

5. (if applicable)


## Data Sources

Include sources (as links) to your datasets. If any of these are different from your sources used in Phase II, please <b>clearly</b> specify.

*   Downloaded Dataset Source: https://nflsavant.com/about.php
*   Web Collection #1 Source: https://www.spotrac.com/nfl/contracts/_/position/qb
*   Web Collection #2 Source: https://gist.github.com/nntrn/ee26cb2a0716de0947a0a4e9a157bc1c 



# Data Analysis
For the Data Analysis section, you are required to utilize your data to complete the following:

*   Create at least 5 insights
*   Generate at least 3 data visualizations

Create a function for each of the following sections mentioned above. Do not forget to fill out the explanation section for each function. 

Make sure your data analysis is not too simple. Performing complex aggregation and using modules not taught in class shows effort, which will increase the chance of receiving full credit. 

# Topic Summary

Please provide a brief executive summary (5 sentences or less) discussing your topic:

Our analysis examines the relationshup between NFL quarterback salaries and their on-field performance to identity valuation inefficiencies. We integrated the "traditional" statistics with an added "clutch factor" to create a quarterback performance index (QPI). This reveals which highly-paid QBs are underperforming and which lower-salaried players deliver elite value. We found that while passing volume nad touchdowns correleate with pay, clutch performance is a significant and overlooked driver of true impact. Ultimately this project provides teams with information about contract negotionations and roster construction to optimize their financial resources. 


## Insights

In [4]:
import pandas as pd
import json
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import plotly.express as px

def insight1():
    salaries = pd.read_csv("salaries.csv")   

    with open("stats.json") as f:
        stats = json.load(f)

    stats_df = pd.DataFrame.from_dict(stats, orient="index").reset_index() #this converts the stats dictionary into a datafrane
    stats_df = stats_df.rename(columns={"index": "Player"})

    qb = stats_df.merge(salaries, on="Player", how="inner")

    clutch = pd.read_csv("clutch_factor.csv")

    qbs = {
        "L.Jackson": "Lamar Jackson", "J.Allen": "Josh Allen", "J.Burrow": "Joe Burrow",
        "J.Goff": "Jared Goff", "J.Daniels": "Jayden Daniels", "B.Mayfield": "Baker Mayfield",
        "P.Mahomes": "Patrick Mahomes", "M.Stafford": "Matthew Stafford", "J.Hurts": "Jalen Hurts",
        "J.Love": "Jordan Love", "J.Herbert": "Justin Herbert", "G.Smith": "Geno Smith",
        "S.Darnold": "Sam Darnold", "B.Purdy": "Brock Purdy", "C.Stroud": "C.J. Stroud",
        "B.Young": "Bryce Young", "K.Murray": "Kyler Murray", "T.Tagovailoa": "Tua Tagovailoa",
        "D.Prescott": "Dak Prescott", "B.Nix": "Bo Nix", "D.Maye": "Drake Maye",
        "A.Rodgers": "Aaron Rodgers", "R.Wilson": "Russell Wilson", "D.Carr": "Derek Carr",
        "K.Cousins": "Kirk Cousins", "T.Lawrence": "Trevor Lawrence", "J.Fields": "Justin Fields",
        "C.Williams": "Caleb Williams", "A.O'Connell": "Aidan O'Connell", "M.Penix": "Michael Penix Jr.",
        "D.Watson": "Deshaun Watson", "T.DeVito": "Tommy DeVito", "C.Wentz": "Carson Wentz",
        "D.Thomas-Robinson": "Dorian Thompson-Robinson", "S.Thompson": "Skylar Thompson",
        "J.Haener": "Jake Haener", "D.Ridder": "Desmond Ridder", "T.Lance": "Trey Lance",
        "B.Zappe": "Bailey Zappe", "B.Allen": "Brandon Allen", "J.Dobbs": "Joshua Dobbs",
        "T.McKee": "Tanner McKee", "K.Pickett": "Kenny Pickett", "J.Garoppolo": "Jimmy Garoppolo",
        "D.Lock": "Drew Lock", "J.Brissett": "Jacoby Brissett", "A.Dalton": "Andy Dalton",
        "M.Jones": "Mac Jones", "M.Rudolph": "Mason Rudolph", "J.Flacco": "Joe Flacco",
        "S.Rattler": "Spencer Rattler", "C.Rush": "Cooper Rush", "D.Jones": "Daniel Jones",
        "G.Minshew": "Gardner Minshew", "A.Richardson": "Anthony Richardson",
        "T.Huntley": "Tyler Huntley", "W.Levis": "Will Levis", "J.Winston": "Jameis Winston",
        "M.Willis": "Malik Willis"
    }

    clutch["Player"] = clutch["Player"].map(qbs)
    clutch = clutch.dropna(subset=["Player"])
    qb = qb.merge(clutch, on="Player", how="left")

    features = ["Passing Yards", "Passing Touchdowns", "Interceptions", "Completion Percentage", "Clutch Score"]
    X = qb[features]
    y = np.log(qb["Salary"])  #log reduces the skewness and stabilizes the variance because there is such high variance in salaries.

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train) #learns the coefficients which best fit y.

    print("Coefficients:", model.coef_)
    features = ["Passing Yards", "Passing Touchdowns", "Interceptions", "Completion Percentage", "Clutch Score"]
    coefficients = model.coef_  

    importance_abs = np.abs(coefficients)

insight1()

Coefficients: [ 0.79231827  0.32397122  0.03234912  0.18863849 -0.19252221]


### Insight 1 Explanation

From this analysis, we can conclude that quarterback salaries are driven primarily by their total production metrics, particularly passing yards and touchdowns, rather than efficiency metrics like completion percentage, clutch performance, or interceptions. Passing yards have the strongest positive effect on salary, indicating that teams value overall output and the ability to move the ball downfield more than situational performance in clutch moments. Touchdowns also positively influence salary, though to a lesser extent, while completion percentage has a modest effect. Interestingly, clutch factor has a slightly negative coefficient, suggesting that situational performance under pressure is not a major determinant of salary in this dataset.

This insight is significant because it highlights how teams prioritize certain types of performance when negotiating or assigning salaries. It suggests that measurable, cumulative statistics (yards, touchdowns) are more influential in compensation than efficiency or situational excellence. For anyone studying the economics of professional football or evaluating player value, this emphasizes the disconnect that can exist between “clutch” performance and financial reward.

To perform this analysis, I had to learn a few new processes. Most importantly, I used feature scaling (StandardScaler) to normalize the statistics so they could be compared on the same scale, and a log transformation of salary to handle the large variation in QB salaries. These steps were necessary to make the linear regression model interpretable and to generate meaningful coefficients that indicate the relative importance of each feature.

In [5]:
def insight2():

    import pandas as pd
    import json
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    import plotly.express as px

    salaries = pd.read_csv("salaries.csv")

    with open("stats.json") as f:
        stats = json.load(f)

    stats_df = pd.DataFrame.from_dict(stats, orient="index").reset_index()
    stats_df = stats_df.rename(columns={"index": "Player"})

    qb = stats_df.merge(salaries, on="Player", how="inner")

    clutch = pd.read_csv("clutch_factor.csv")

    qbs = {
        "L.Jackson": "Lamar Jackson",
        "J.Allen": "Josh Allen",
        "J.Burrow": "Joe Burrow",
        "J.Goff": "Jared Goff",
        "J.Daniels": "Jayden Daniels",
        "B.Mayfield": "Baker Mayfield",
        "P.Mahomes": "Patrick Mahomes",
        "M.Stafford": "Matthew Stafford",
        "J.Hurts": "Jalen Hurts",
        "J.Love": "Jordan Love",
        "J.Herbert": "Justin Herbert",
        "G.Smith": "Geno Smith",
        "S.Darnold": "Sam Darnold",
        "B.Purdy": "Brock Purdy",
        "C.Stroud": "C.J. Stroud",
        "B.Young": "Bryce Young",
        "K.Murray": "Kyler Murray",
        "T.Tagovailoa": "Tua Tagovailoa",
        "D.Prescott": "Dak Prescott",
        "B.Nix": "Bo Nix",
        "D.Maye": "Drake Maye",
        "A.Rodgers": "Aaron Rodgers",
        "R.Wilson": "Russell Wilson",
        "D.Carr": "Derek Carr",
        "K.Cousins": "Kirk Cousins",
        "T.Lawrence": "Trevor Lawrence",
        "J.Fields": "Justin Fields",
        "C.Williams": "Caleb Williams",
        "A.O'Connell": "Aidan O'Connell",
        "M.Penix": "Michael Penix Jr.",
        "D.Watson": "Deshaun Watson",
        "T.DeVito": "Tommy DeVito",
        "C.Wentz": "Carson Wentz",
        "D.Thomas-Robinson": "Dorian Thompson-Robinson",
        "S.Thompson": "Skylar Thompson",
        "J.Haener": "Jake Haener",
        "D.Ridder": "Desmond Ridder",
        "T.Lance": "Trey Lance",
        "B.Zappe": "Bailey Zappe",
        "B.Allen": "Brandon Allen",
        "J.Dobbs": "Joshua Dobbs",
        "T.McKee": "Tanner McKee",
        "K.Pickett": "Kenny Pickett",
        "J.Garoppolo": "Jimmy Garoppolo",
        "D.Lock": "Drew Lock",
        "J.Brissett": "Jacoby Brissett",
        "A.Dalton": "Andy Dalton",
        "M.Jones": "Mac Jones",
        "M.Rudolph": "Mason Rudolph",
        "J.Flacco": "Joe Flacco",
        "S.Rattler": "Spencer Rattler",
        "C.Rush": "Cooper Rush",
        "D.Jones": "Daniel Jones",
        "G.Minshew": "Gardner Minshew",
        "A.Richardson": "Anthony Richardson",
        "T.Huntley": "Tyler Huntley",
        "W.Levis": "Will Levis",
        "J.Winston": "Jameis Winston",
        "M.Willis": "Malik Willis"
    }

    clutch["Player"] = clutch["Player"].map(qbs)
    clutch = clutch.dropna(subset=["Player"])

    qb = qb.merge(clutch, on="Player", how="left")

    features = [
        "Passing Yards",
        "Rushing Yards",
        "Completion Percentage",
        "Passing Touchdowns",
        "Interceptions",
        "Clutch Score"
    ]

    X = qb[features]


    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=3, random_state=42)
    qb["Cluster"] = kmeans.fit_predict(X_scaled)

    cluster_summary = qb.groupby('Cluster')[features + ['Salary']].mean().round(2)
    cluster_summary['Num_QBs'] = qb.groupby('Cluster').size()
    print("Cluster Summary Statistics:")
    print(cluster_summary)

    cluster_counts = qb['Cluster'].value_counts().sort_index()
    print("\nNumber of QBs per Cluster:")
    print(cluster_counts)

insight2()

Cluster Summary Statistics:
         Passing Yards  Rushing Yards  Completion Percentage  \
Cluster                                                        
0              2653.90         146.25                  64.63   
1               685.00          60.18                  61.92   
2              4036.77         439.85                  68.27   

         Passing Touchdowns  Interceptions  Clutch Score       Salary  Num_QBs  
Cluster                                                                         
0                     16.00           8.80         28.08  25227608.20       20  
1                      3.32           2.18         26.60   4967841.68       22  
2                     29.62          10.31         33.64  32958708.23       13  

Number of QBs per Cluster:
Cluster
0    20
1    22
2    13
Name: count, dtype: int64


c:\Users\adith\anaconda4\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


### Insight 2 Explanation

KMeans clustering on six key performance metrics identified three meaningful quarterback archetypes that provide a framework for NFL decision-making. Cluster 0 (20 QBs) represents traditional pocket passers who offer steady, efficient production with minimal rushing impact at a mid-tier salary of $25M. Cluster 1 (22 QBs) consists of backups and low-volume players with minimal output which again shows thelow $5M salaries. Most significantly, Cluster 2 (13 QBs) has elite dual-threat franchise quarterbacks who get a premium $33M salaries for their exceptional combination of high-volume passing and substantial rushing production, despite carrying higher interception risk. This clustering enables teams to benchmark contract value—identifying overpaid or undervalued quarterbacks relative to their cluster peers—while also informing roster construction by matching archetypes to offensive schemes. Defensively, it directly shapes game planning, as facing a Cluster 2 quarterback requires containing both passing and rushing threats, whereas defending Cluster 0 quarterbacks allows greater focus on stopping the run. The analysis confirms that the modern NFL's most valuable quarterbacks are those who can stress defenses through both parts of the game, though teams must weigh this dual-threat premium against the increased turnover risk that comes with it.

In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
import json

def insight_3():
    salaries = pd.read_csv("salaries.csv")   

    with open("stats.json") as f:
        stats = json.load(f)

    stats_df = pd.DataFrame.from_dict(stats, orient="index").reset_index()
    stats_df = stats_df.rename(columns={"index": "Player"})

    qb = stats_df.merge(salaries, on="Player", how="inner")

    clutch = pd.read_csv("clutch_factor.csv")

    qbs = {
        "L.Jackson": "Lamar Jackson", "J.Allen": "Josh Allen", "J.Burrow": "Joe Burrow",
        "J.Goff": "Jared Goff", "J.Daniels": "Jayden Daniels", "B.Mayfield": "Baker Mayfield",
        "P.Mahomes": "Patrick Mahomes", "M.Stafford": "Matthew Stafford", "J.Hurts": "Jalen Hurts",
        "J.Love": "Jordan Love", "J.Herbert": "Justin Herbert", "G.Smith": "Geno Smith",
        "S.Darnold": "Sam Darnold", "B.Purdy": "Brock Purdy", "C.Stroud": "C.J. Stroud",
        "B.Young": "Bryce Young", "K.Murray": "Kyler Murray", "T.Tagovailoa": "Tua Tagovailoa",
        "D.Prescott": "Dak Prescott", "B.Nix": "Bo Nix", "D.Maye": "Drake Maye",
        "A.Rodgers": "Aaron Rodgers", "R.Wilson": "Russell Wilson", "D.Carr": "Derek Carr",
        "K.Cousins": "Kirk Cousins", "T.Lawrence": "Trevor Lawrence", "J.Fields": "Justin Fields",
        "C.Williams": "Caleb Williams", "A.O'Connell": "Aidan O'Connell", "M.Penix": "Michael Penix Jr.",
        "D.Watson": "Deshaun Watson", "T.DeVito": "Tommy DeVito", "C.Wentz": "Carson Wentz",
        "D.Thomas-Robinson": "Dorian Thompson-Robinson", "S.Thompson": "Skylar Thompson",
        "J.Haener": "Jake Haener", "D.Ridder": "Desmond Ridder", "T.Lance": "Trey Lance",
        "B.Zappe": "Bailey Zappe", "B.Allen": "Brandon Allen", "J.Dobbs": "Joshua Dobbs",
        "T.McKee": "Tanner McKee", "K.Pickett": "Kenny Pickett", "J.Garoppolo": "Jimmy Garoppolo",
        "D.Lock": "Drew Lock", "J.Brissett": "Jacoby Brissett", "A.Dalton": "Andy Dalton",
        "M.Jones": "Mac Jones", "M.Rudolph": "Mason Rudolph", "J.Flacco": "Joe Flacco",
        "S.Rattler": "Spencer Rattler", "C.Rush": "Cooper Rush", "D.Jones": "Daniel Jones",
        "G.Minshew": "Gardner Minshew", "A.Richardson": "Anthony Richardson",
        "T.Huntley": "Tyler Huntley", "W.Levis": "Will Levis", "J.Winston": "Jameis Winston",
        "M.Willis": "Malik Willis"
    }

    # name inc.
    clutch["Player"] = clutch["Player"].map(qbs)
    clutch = clutch.dropna(subset=["Player"])

    merged = qb.merge(clutch, on="Player", how="left")

    merged["pyd_z"] = (merged["Passing Yards"] - merged["Passing Yards"].mean()) / merged["Passing Yards"].std()
    merged["comp_z"] = (merged["Completion Percentage"] - merged["Completion Percentage"].mean()) / merged["Completion Percentage"].std()
    merged["ptd_z"] = (merged["Passing Touchdowns"] - merged["Passing Touchdowns"].mean()) / merged["Passing Touchdowns"].std()
    merged["int_z"] = (merged["Interceptions"] - merged["Interceptions"].mean()) / merged["Interceptions"].std()
    merged["ryd_z"] = (merged["Rushing Yards"] - merged["Rushing Yards"].mean()) / merged["Rushing Yards"].std()
    merged["rtd_z"] = (merged["Rushing Touchdowns"] - merged["Rushing Touchdowns"].mean()) / merged["Rushing Touchdowns"].std()
    merged["clutch_z"] = (merged["Clutch Score"] - merged["Clutch Score"].mean()) / merged["Clutch Score"].std()

    merged["score_numerator"] = (
        0.15*merged["pyd_z"] + 0.15*merged["comp_z"] + 0.25*merged["ptd_z"] 
        - 0.20*merged["int_z"] + 0.05*merged["rtd_z"] + 0.05*merged["ryd_z"] + 0.15*merged["clutch_z"]
    )

    merged["w_salary"] = 1 + 0.0000001 * merged["Salary"]
    merged["QPI"] = (merged["score_numerator"] / merged["w_salary"]).round(4)

    top_undervalued = merged.sort_values(by="QPI", ascending=False).head(10)
    top_overvalued = merged.sort_values(by="QPI").head(10)

    print("\nTop 5 Undervalued QBs")
    print(top_undervalued[["Player", "Salary", "QPI"]].to_string(index=False))

    print("\nTop 5 Overvalued QBs")
    print(top_overvalued[["Player", "Salary", "QPI"]].to_string(index=False))

insight_3()



Top 5 Undervalued QBs
        Player   Salary    QPI
Jayden Daniels  9436663 0.3645
        Bo Nix  4653292 0.3101
   Brock Purdy   934252 0.2802
   Sam Darnold 10000000 0.2581
  Malik Willis  1290025 0.2571
 Lamar Jackson 52000000 0.2229
Baker Mayfield 33333333 0.2188
  Joshua Dobbs  2250000 0.1915
    Joe Burrow 55000000 0.1804
    Josh Allen 43005667 0.1584

Top 5 Overvalued QBs
         Player  Salary     QPI
    Jake Haener 1136204 -0.8928
Spencer Rattler 1089120 -0.7030
   Bailey Zappe  985000 -0.6885
   Carson Wentz 3325000 -0.5306
  Brandon Allen 2020000 -0.4993
 Desmond Ridder  985000 -0.4021
  Tyler Huntley 1125000 -0.3780
     Will Levis 2385541 -0.3639
     Trey Lance 8526319 -0.3497
  Kenny Pickett 3516976 -0.3259


### Insight 3 Explanation

This insight creates our Quarterback Performance Index (QPI), an all in one metric that evaluates quarterbacks comprehensively. To calculate this statistic, we first find a weighted sum of each quarterback's normalized passing yards, completion percentage, passing touchdowns, interceptions, rushing yards, rushing touchdowns, and clutch score (another metric we created). The weights are assigned based on our perception of how valuable each are in deriving a cumulative statistic. The statistics are normalized to account for different playstyles and fairly compare quarterbacks. Then, we divided that weighted sum by a very small factor of each quarterback's salary. The '1 + 0.0000001 * merged["Salary"]' is done to ensure that salary does not play too large of a role in QPI, as otherwise rookies and low contract players would be valued too much, even if they performed poorly. Ultimately, this statistic allows us to figure out which quarterbacks are undervalued and overvalued in regard to their salary. For an NFL GM who not only has to build the best possible team but also navigate cap constraints, having quarterback salary information integrated into a statistic is extremely useful. To build this insight, we merged our three main datasets (clutch factor, individual statistics, and salary). Overall, our QPI insight serves as our culminating measure of our project, as it offers a single index that captures a quarterback's total impact through statistics, context based performance (clutch ability), and salary.

In [7]:
import pandas as pd
import numpy as np
import plotly.express as px
import json

def insight4():
    salaries = pd.read_csv("salaries.csv")   

    with open("stats.json") as f:
        stats = json.load(f)

    stats_df = pd.DataFrame.from_dict(stats, orient="index").reset_index()
    stats_df = stats_df.rename(columns={"index": "Player"})

    qb = stats_df.merge(salaries, on="Player", how="inner")

    clutch = pd.read_csv("clutch_factor.csv")

    qbs = {
        "L.Jackson": "Lamar Jackson",
        "J.Allen": "Josh Allen",
        "J.Burrow": "Joe Burrow",
        "J.Goff": "Jared Goff",
        "J.Daniels": "Jayden Daniels",
        "B.Mayfield": "Baker Mayfield",
        "P.Mahomes": "Patrick Mahomes",
        "M.Stafford": "Matthew Stafford",
        "J.Hurts": "Jalen Hurts",
        "J.Love": "Jordan Love",
        "J.Herbert": "Justin Herbert",
        "G.Smith": "Geno Smith",
        "S.Darnold": "Sam Darnold",
        "B.Purdy": "Brock Purdy",
        "C.Stroud": "C.J. Stroud",
        "B.Young": "Bryce Young",
        "K.Murray": "Kyler Murray",
        "T.Tagovailoa": "Tua Tagovailoa",
        "D.Prescott": "Dak Prescott",
        "B.Nix": "Bo Nix",
        "D.Maye": "Drake Maye",
        "A.Rodgers": "Aaron Rodgers",
        "R.Wilson": "Russell Wilson",
        "D.Carr": "Derek Carr",
        "K.Cousins": "Kirk Cousins",
        "T.Lawrence": "Trevor Lawrence",
        "J.Fields": "Justin Fields",
        "C.Williams": "Caleb Williams",
        "A.O'Connell": "Aidan O'Connell",
        "M.Penix": "Michael Penix Jr.",
        "D.Watson": "Deshaun Watson",
        "T.DeVito": "Tommy DeVito",
        "C.Wentz": "Carson Wentz",
        "D.Thomas-Robinson": "Dorian Thompson-Robinson",
        "S.Thompson": "Skylar Thompson",
        "J.Haener": "Jake Haener",
        "D.Ridder": "Desmond Ridder",
        "T.Lance": "Trey Lance",
        "B.Zappe": "Bailey Zappe",
        "B.Allen": "Brandon Allen",
        "J.Dobbs": "Joshua Dobbs",
        "T.McKee": "Tanner McKee",
        "K.Pickett": "Kenny Pickett",
        "J.Garoppolo": "Jimmy Garoppolo",
        "D.Lock": "Drew Lock",
        "J.Brissett": "Jacoby Brissett",
        "A.Dalton": "Andy Dalton",
        "M.Jones": "Mac Jones",
        "M.Rudolph": "Mason Rudolph",
        "J.Flacco": "Joe Flacco",
        "S.Rattler": "Spencer Rattler",
        "C.Rush": "Cooper Rush",
        "D.Jones": "Daniel Jones",
        "G.Minshew": "Gardner Minshew",
        "A.Richardson": "Anthony Richardson",
        "T.Huntley": "Tyler Huntley",
        "W.Levis": "Will Levis",
        "J.Winston": "Jameis Winston",
        "M.Willis": "Malik Willis"
    }

    # name inconsistencies
    clutch["Player"] = clutch["Player"].map(qbs)
    clutch = clutch.dropna(subset=["Player"])

    merged = qb.merge(clutch, on="Player", how="left")

    merged["Touchdown/TO"] = np.where(
        merged["Interceptions"] != 0,
        merged["Passing Touchdowns"] / merged["Interceptions"],
        0
    )

    fig = px.scatter(
        merged, 
        x="Player", 
        y="Touchdown/TO", 
        title="QBs Ranked By Touchdown/TO", 
        color="Touchdown/TO", 
        color_continuous_scale="Viridis"
    )
    fig.show()

    top_5_td_to = merged[['Player', 'Touchdown/TO']].sort_values(by='Touchdown/TO', ascending=False).head(5)
    print("\nTop 5 QBs by Touchdown-to-Interception ratio:")
    print(top_5_td_to.to_string(index=False))

    avg_td_to = merged["Touchdown/TO"].mean()
    print(f"\nAverage Touchdown-to-Interception ratio across all QBs: {avg_td_to:.2f}")

    qb_zero_int = merged[merged["Interceptions"] == 0]
    print(f"\nNumber of QBs with zero interceptions: {len(qb_zero_int)}")
insight4()



Top 5 QBs by Touchdown-to-Interception ratio:
        Player  Touchdown/TO
 Lamar Jackson     10.250000
Justin Herbert      7.666667
 Justin Fields      5.000000
    Joe Burrow      4.777778
    Josh Allen      4.666667

Average Touchdown-to-Interception ratio across all QBs: 2.06

Number of QBs with zero interceptions: 4


### Insight 4 Explanation

This insight calculates and ranks quarterbacks based on their touchdown to interception ratio, one of the clearest indicators of a quarterback's decision making and ability to score while limiting mistakes. NFL GM's typically look for quarterbacks they can consistently rely on, and a high touchdown to interception ratio indicates exactly that, as teams that win the turnover battle usually win each week. The reason we chose to include this insight is because it avoids yardage totals, which, due to checkdowns and elite wide receivers who have a high number of yards after catch (YAC), are often inflated. Instead, this metric directly evaluates a quarterback's good plays versus bad plays. Despite this insight's simplicity, it's extremely meaningful as it provides a different facet of quarterback evaluation while avoiding the potential pitfalls of a yardage based metric like our QPI. In calculating this metric, we had to handle edge cases such as quarterbacks having no interceptions, usually due to them being backups with less playing time.

In [8]:
import pandas as pd
import numpy as np
import plotly.express as px

def insight5():
    df = pd.read_csv("new-pbp.csv")
    salariesdf = pd.read_csv("salaries.csv")

    players_dict = {
        "L.Jackson": "BAL", "J.Allen": "BUF", "J.Burrow": "CIN", "J.Goff": "DET", "J.Daniels": "WAS", 
        "B.Mayfield": "TB", "P.Mahomes": "KC", "M.Stafford": "LA", "J.Hurts": "PHI", "J.Love": "GB", 
        "J.Herbert": "LAC", "G.Smith": "SEA", "S.Darnold": "MIN", "B.Purdy": "SF", "C.Stroud": "HOU", 
        "B.Young": "CAR", "K.Murray": "ARI", "T.Tagovailoa": "MIA", "D.Prescott": "DAL", "B.Nix": "DEN", 
        "D.Maye": "NE", "A.Rodgers": "NYJ", "R.Wilson": "PIT", "D.Carr": "NO", "K.Cousins": "ATL", 
        "T.Lawrence": "JAX", "J.Fields": "PIT", "C.Williams": "CHI", "A.O'Connell": "LV", "M.Penix": "ATL", 
        "D.Watson": "CLE", "T.DeVito": "NYG", "C.Wentz": "KC", "D.Thompson-Robinson": "CLE", "S.Thompson": "MIA", 
        "J.Haener": "NO", "D.Ridder": "LV", "T.Lance": "DAL", "B.Zappe": "CLE", "B.Allen": "SF", 
        "J.Dobbs": "SF", "T.McKee": "PHI", "K.Pickett": "PHI", "J.Garoppolo": "LA", "D.Lock": "NYG", 
        "J.Brissett": "NE", "A.Dalton": "CAR", "M.Jones": "JAX", "M.Rudolph": "TEN", "J.Flacco": "IND", 
        "S.Rattler": "NO", "C.Rush": "DAL", "D.Jones": "NYG", "G.Minshew": "LV", "A.Richardson": "IND", 
        "T.Huntley": "MIA", "W.Levis": "TEN", "J.Winston": "CLE", "M.Willis": "GB"
    }

    # team to qb
    team_to_qbs = {}
    for player, team in players_dict.items():
        if team not in team_to_qbs:
            team_to_qbs[team] = []
        team_to_qbs[team].append(player)

    td_counts = {}
    for player in players_dict.keys():
        team = players_dict[player]
        player_upper = player.upper()
        df_player = df[
            (df["OffenseTeam"] == team) &
            (df["YardLineDirection"] == "OPP") &
            (df["YardLine"] <= 20) &
            (df["IsTouchdown"] == 1) &
            (df["Description"].str.contains(player_upper, na=False))
        ]
        td_counts[player] = len(df_player)

    num_redzone_possessions = {player: 0 for player in players_dict}
    current_team = None
    in_redzone = False
    current_qb = None

    for _, row in df.iterrows():
        if row["OffenseTeam"] != current_team:
            current_team = row["OffenseTeam"]
            in_redzone = False
            current_qb = None

        desc_upper = row["Description"].upper()
        for qb in team_to_qbs[current_team]:
            if qb.upper() in desc_upper:
                current_qb = qb
                break

        if (row["YardLineDirection"] == "OPP") and (row["YardLine"] <= 20) and not in_redzone:
            in_redzone = True
            if current_qb is not None:
                num_redzone_possessions[current_qb] += 1
        else:
            in_redzone = False

    redzone_percent = {}
    for player in players_dict.keys():
        trips = num_redzone_possessions[player]
        tds = td_counts[player]
        redzone_percent[player] = round((tds / trips) * 100, 2) if trips > 0 else 0

    third_down_success_rate = {}
    for team, qbs in team_to_qbs.items():
        for qb in qbs:
            df_third = df[
                (df["IsPenalty"] == 0) &
                (df["Down"] == 3) &
                (df["PlayType"] != "RUSH") &
                (df["OffenseTeam"] == team) &
                (df["PlayType"] != "NO PLAY") &
                (df["Description"].str.contains(qb.upper(), na=False, regex=False))
            ]
            df_success = df_third[df_third["SeriesFirstDown"] == 1]
            third_down_success_rate[qb] = round((len(df_success) / len(df_third)) * 100, 2) if len(df_third) > 0 else 0

    clutch_score = {}
    for player in players_dict.keys():
        clutch_score[player] = round((redzone_percent[player] * 0.35) + (third_down_success_rate[player] * 0.65), 2)

    clutch_df = pd.DataFrame(clutch_score.items(), columns=["Player", "Clutch Score"])
    clutch_df.to_csv("clutch_factor.csv", index=False)

    fig = px.bar(
        clutch_df,
        x="Player",
        y="Clutch Score",
        title="QB Clutch Score",
        color="Clutch Score",
        color_continuous_scale="Viridis"
    )
    fig.show()

    top_5_clutch = clutch_df.sort_values(by="Clutch Score", ascending=False).head(5)
    print("\nTop 5 QBs by Clutch Score:")
    print(top_5_clutch.to_string(index=False))
    high_third_down = [player for player, rate in third_down_success_rate.items() if rate > 50]
    print(f"\nQBs with >70% third-down conversion rate: {high_third_down}")
insight5()


Top 5 QBs by Clutch Score:
     Player  Clutch Score
    J.Dobbs         44.17
 B.Mayfield         39.76
J.Garoppolo         39.67
  L.Jackson         37.26
   J.Burrow         36.48

QBs with >70% third-down conversion rate: ['J.Garoppolo']


### Insight 5 Explanation

The purpose of this code is to create a "Clutch Score" for each quarterback, allowing us to rank and evaluate quarterbacks purely on their ability to perform when it matters most. It's often said that the difference between a good quarterback and a great one is their ability to be clutch, and this insight focuses specifically on that. From the perspective of an NFL GM, this insight is valuable when comparing quarterbacks or determining if a particular quarterback is worth signing. The score is calculated as a weighted sum of the percentage of red zone possessions in which the quarterback makes a scoring play and the percentage of third downs converted by the quarterback. Both of those areas (third down and red zone conversion) are critical because they provide context based insight that raw statistics cannot. While this insight builds towards our broader QPI metric, it in itself serves as a meaningful measure for ranking quarterbacks. The 0.35 and 0.65 weights are assigned based on our own perception of the relative importance of red zone and third down conversion rates. In calculating this metric, we had to manipulate the dataframe to include the specific situations (red zone and third downs) we wanted and ensure that red zone possessions were not overcounted.

## Data Visualizations

In [9]:
import pandas as pd
import json
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import plotly.express as px


def visual1():
    salaries = pd.read_csv("salaries.csv")
    with open("stats.json") as f:
        stats = json.load(f)
    stats_df = pd.DataFrame.from_dict(stats, orient="index").reset_index()
    stats_df = stats_df.rename(columns={"index": "Player"})
    
    clutch = pd.read_csv("clutch_factor.csv")
    
    qbs = {
        "L.Jackson": "Lamar Jackson",
        "J.Allen": "Josh Allen",
        "J.Burrow": "Joe Burrow",
        "J.Goff": "Jared Goff",
        "J.Daniels": "Jayden Daniels",
        "B.Mayfield": "Baker Mayfield",
        "P.Mahomes": "Patrick Mahomes",
        "M.Stafford": "Matthew Stafford",
        "J.Hurts": "Jalen Hurts",
        "J.Love": "Jordan Love",
        "J.Herbert": "Justin Herbert",
        "G.Smith": "Geno Smith",
        "S.Darnold": "Sam Darnold",
        "B.Purdy": "Brock Purdy",
        "C.Stroud": "C.J. Stroud",
        "B.Young": "Bryce Young",
        "K.Murray": "Kyler Murray",
        "T.Tagovailoa": "Tua Tagovailoa",
        "D.Prescott": "Dak Prescott",
        "B.Nix": "Bo Nix",
        "D.Maye": "Drake Maye",
        "A.Rodgers": "Aaron Rodgers",
        "R.Wilson": "Russell Wilson",
        "D.Carr": "Derek Carr",
        "K.Cousins": "Kirk Cousins",
        "T.Lawrence": "Trevor Lawrence",
        "J.Fields": "Justin Fields",
        "C.Williams": "Caleb Williams",
        "A.O'Connell": "Aidan O'Connell",
        "M.Penix": "Michael Penix Jr.",
        "D.Watson": "Deshaun Watson",
        "T.DeVito": "Tommy DeVito",
        "C.Wentz": "Carson Wentz",
        "D.Thomas-Robinson": "Dorian Thompson-Robinson",
        "S.Thompson": "Skylar Thompson",
        "J.Haener": "Jake Haener",
        "D.Ridder": "Desmond Ridder",
        "T.Lance": "Trey Lance",
        "B.Zappe": "Bailey Zappe",
        "B.Allen": "Brandon Allen",
        "J.Dobbs": "Joshua Dobbs",
        "T.McKee": "Tanner McKee",
        "K.Pickett": "Kenny Pickett",
        "J.Garoppolo": "Jimmy Garoppolo",
        "D.Lock": "Drew Lock",
        "J.Brissett": "Jacoby Brissett",
        "A.Dalton": "Andy Dalton",
        "M.Jones": "Mac Jones",
        "M.Rudolph": "Mason Rudolph",
        "J.Flacco": "Joe Flacco",
        "S.Rattler": "Spencer Rattler",
        "C.Rush": "Cooper Rush",
        "D.Jones": "Daniel Jones",
        "G.Minshew": "Gardner Minshew",
        "A.Richardson": "Anthony Richardson",
        "T.Huntley": "Tyler Huntley",
        "W.Levis": "Will Levis",
        "J.Winston": "Jameis Winston",
        "M.Willis": "Malik Willis"
    }
    
    clutch["Player"] = clutch["Player"].map(qbs)
    clutch = clutch.dropna(subset=["Player"])
    
    qb = stats_df.merge(salaries, on="Player", how="inner").merge(clutch, on="Player", how="left")
    
    X = qb[["Passing Yards", "Passing Touchdowns", "Interceptions", "Completion Percentage", "Clutch Score"]]
    y = np.log(qb["Salary"])
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)    
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    fig1 = px.scatter(
        x=y_test,
        y=y_pred,
        labels={"x": "Actual Salary", "y": "Predicted Salary"},
        title="Predicted vs Actual Salary"
    )
    fig1.add_shape(
        type="line",
        x0=y_test.min(),
        y0=y_test.min(),
        x1=y_test.max(),
        y1=y_test.max(),
        line=dict(color="red", width=2)
    )
    fig1.show()
    
    features = X.columns
    coefficients = model.coef_
    importance_abs = np.abs(coefficients)
    
    fig2 = px.pie(
        names=features,
        values=importance_abs,
        title="Feature Importance (Magnitude)"
    )
    fig2.show()
visual1()


### Visualization 1 Explanation

Based on the magnitude of feature importance analaysis, NFL quarterback salaries are overwhelmingly determined by volume production rather than efficiency or risk management. Passing yards alone account for 51.8% of the salary determination, which shows that it is the primary driver of compensation. Passing touchdowns contribute another 21.2%, showing that scoring production serves as the secondary consideration. The efficiency metrics like clutch performance and completion perctnage just account for one quarter of the influence. One thing that should be noted is that the clutch performance actually drives negatively in salary which shows that the system is favored much more towards production than efficiency. Most suprisingly, the interceptions have a negligible impact, which shows that teams are willing to tolerate turnovers from high-volume passers. 

In [10]:
import pandas as pd
import json
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.express as px

def visual2():
    salaries = pd.read_csv("salaries.csv")
    with open("stats.json") as f:
        stats = json.load(f)
    stats_df = pd.DataFrame.from_dict(stats, orient="index").reset_index()
    stats_df = stats_df.rename(columns={"index": "Player"})
    
    clutch = pd.read_csv("clutch_factor.csv")
    
    qbs = {
        "L.Jackson": "Lamar Jackson",
        "J.Allen": "Josh Allen",
        "J.Burrow": "Joe Burrow",
        "J.Goff": "Jared Goff",
        "J.Daniels": "Jayden Daniels",
        "B.Mayfield": "Baker Mayfield",
        "P.Mahomes": "Patrick Mahomes",
        "M.Stafford": "Matthew Stafford",
        "J.Hurts": "Jalen Hurts",
        "J.Love": "Jordan Love",
        "J.Herbert": "Justin Herbert",
        "G.Smith": "Geno Smith",
        "S.Darnold": "Sam Darnold",
        "B.Purdy": "Brock Purdy",
        "C.Stroud": "C.J. Stroud",
        "B.Young": "Bryce Young",
        "K.Murray": "Kyler Murray",
        "T.Tagovailoa": "Tua Tagovailoa",
        "D.Prescott": "Dak Prescott",
        "B.Nix": "Bo Nix",
        "D.Maye": "Drake Maye",
        "A.Rodgers": "Aaron Rodgers",
        "R.Wilson": "Russell Wilson",
        "D.Carr": "Derek Carr",
        "K.Cousins": "Kirk Cousins",
        "T.Lawrence": "Trevor Lawrence",
        "J.Fields": "Justin Fields",
        "C.Williams": "Caleb Williams",
        "A.O'Connell": "Aidan O'Connell",
        "M.Penix": "Michael Penix Jr.",
        "D.Watson": "Deshaun Watson",
        "T.DeVito": "Tommy DeVito",
        "C.Wentz": "Carson Wentz",
        "D.Thomas-Robinson": "Dorian Thompson-Robinson",
        "S.Thompson": "Skylar Thompson",
        "J.Haener": "Jake Haener",
        "D.Ridder": "Desmond Ridder",
        "T.Lance": "Trey Lance",
        "B.Zappe": "Bailey Zappe",
        "B.Allen": "Brandon Allen",
        "J.Dobbs": "Joshua Dobbs",
        "T.McKee": "Tanner McKee",
        "K.Pickett": "Kenny Pickett",
        "J.Garoppolo": "Jimmy Garoppolo",
        "D.Lock": "Drew Lock",
        "J.Brissett": "Jacoby Brissett",
        "A.Dalton": "Andy Dalton",
        "M.Jones": "Mac Jones",
        "M.Rudolph": "Mason Rudolph",
        "J.Flacco": "Joe Flacco",
        "S.Rattler": "Spencer Rattler",
        "C.Rush": "Cooper Rush",
        "D.Jones": "Daniel Jones",
        "G.Minshew": "Gardner Minshew",
        "A.Richardson": "Anthony Richardson",
        "T.Huntley": "Tyler Huntley",
        "W.Levis": "Will Levis",
        "J.Winston": "Jameis Winston",
        "M.Willis": "Malik Willis"
    }
    
    clutch["Player"] = clutch["Player"].map(qbs)
    clutch = clutch.dropna(subset=["Player"])
    
    qb = stats_df.merge(salaries, on="Player", how="inner").merge(clutch, on="Player", how="left")
    
    features = [
        "Passing Yards",
        "Rushing Yards",
        "Completion Percentage",
        "Passing Touchdowns",
        "Interceptions",
        "Clutch Score"
    ]
    X = qb[features]
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    kmeans = KMeans(n_clusters=3, random_state=42)
    qb["Cluster"] = kmeans.fit_predict(X_scaled)
    
    fig = px.scatter(
        qb,
        x="Passing Yards",
        y="Rushing Yards",
        color="Cluster",
        size="Passing Touchdowns",
        hover_data=["Player", "Salary", "Completion Percentage", "Interceptions", "Clutch Score"],
        title="QB Clustering: Passing vs Rushing"
    )
    fig.show()

visual2()


c:\Users\adith\anaconda4\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning:

KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.



### Visualization 2 Explanation

This shows three distinct types of quarterbacks based on their playing style and performance. The pink cluster are the backups and low volume players that have minimal statistical output. Based on their sizes too, they have limited passing touchdowns. The blue cluster are the traditional pocket passers who operate efficiently without extreme highs in a single category. They have moderate passing production but limited rushing contribution as you can see they are lower on the y axis. The yellow cluster is the premier cluster of franchise QBs as they are not just elite passers but based on the visual, they have high amount of rushing yards. These clusters create a clear benchmark for quarterback compensation as teams can now target a specific archetype that fits their offensive philosophy. 

In [11]:
import pandas as pd
import numpy as np
import plotly.express as px
import json

def insight_3():
    salaries = pd.read_csv("salaries.csv")   

    with open("stats.json") as f:
        stats = json.load(f)

    stats_df = pd.DataFrame.from_dict(stats, orient="index").reset_index()
    stats_df = stats_df.rename(columns={"index": "Player"})

    qb = stats_df.merge(salaries, on="Player", how="inner")

    clutch = pd.read_csv("clutch_factor.csv")

    qbs = {
        "L.Jackson": "Lamar Jackson", "J.Allen": "Josh Allen", "J.Burrow": "Joe Burrow",
        "J.Goff": "Jared Goff", "J.Daniels": "Jayden Daniels", "B.Mayfield": "Baker Mayfield",
        "P.Mahomes": "Patrick Mahomes", "M.Stafford": "Matthew Stafford", "J.Hurts": "Jalen Hurts",
        "J.Love": "Jordan Love", "J.Herbert": "Justin Herbert", "G.Smith": "Geno Smith",
        "S.Darnold": "Sam Darnold", "B.Purdy": "Brock Purdy", "C.Stroud": "C.J. Stroud",
        "B.Young": "Bryce Young", "K.Murray": "Kyler Murray", "T.Tagovailoa": "Tua Tagovailoa",
        "D.Prescott": "Dak Prescott", "B.Nix": "Bo Nix", "D.Maye": "Drake Maye",
        "A.Rodgers": "Aaron Rodgers", "R.Wilson": "Russell Wilson", "D.Carr": "Derek Carr",
        "K.Cousins": "Kirk Cousins", "T.Lawrence": "Trevor Lawrence", "J.Fields": "Justin Fields",
        "C.Williams": "Caleb Williams", "A.O'Connell": "Aidan O'Connell", "M.Penix": "Michael Penix Jr.",
        "D.Watson": "Deshaun Watson", "T.DeVito": "Tommy DeVito", "C.Wentz": "Carson Wentz",
        "D.Thomas-Robinson": "Dorian Thompson-Robinson", "S.Thompson": "Skylar Thompson",
        "J.Haener": "Jake Haener", "D.Ridder": "Desmond Ridder", "T.Lance": "Trey Lance",
        "B.Zappe": "Bailey Zappe", "B.Allen": "Brandon Allen", "J.Dobbs": "Joshua Dobbs",
        "T.McKee": "Tanner McKee", "K.Pickett": "Kenny Pickett", "J.Garoppolo": "Jimmy Garoppolo",
        "D.Lock": "Drew Lock", "J.Brissett": "Jacoby Brissett", "A.Dalton": "Andy Dalton",
        "M.Jones": "Mac Jones", "M.Rudolph": "Mason Rudolph", "J.Flacco": "Joe Flacco",
        "S.Rattler": "Spencer Rattler", "C.Rush": "Cooper Rush", "D.Jones": "Daniel Jones",
        "G.Minshew": "Gardner Minshew", "A.Richardson": "Anthony Richardson",
        "T.Huntley": "Tyler Huntley", "W.Levis": "Will Levis", "J.Winston": "Jameis Winston",
        "M.Willis": "Malik Willis"
    }

    # name inc.
    clutch["Player"] = clutch["Player"].map(qbs)
    clutch = clutch.dropna(subset=["Player"])

    merged = qb.merge(clutch, on="Player", how="left")

    merged["pyd_z"] = (merged["Passing Yards"] - merged["Passing Yards"].mean()) / merged["Passing Yards"].std()
    merged["comp_z"] = (merged["Completion Percentage"] - merged["Completion Percentage"].mean()) / merged["Completion Percentage"].std()
    merged["ptd_z"] = (merged["Passing Touchdowns"] - merged["Passing Touchdowns"].mean()) / merged["Passing Touchdowns"].std()
    merged["int_z"] = (merged["Interceptions"] - merged["Interceptions"].mean()) / merged["Interceptions"].std()
    merged["ryd_z"] = (merged["Rushing Yards"] - merged["Rushing Yards"].mean()) / merged["Rushing Yards"].std()
    merged["rtd_z"] = (merged["Rushing Touchdowns"] - merged["Rushing Touchdowns"].mean()) / merged["Rushing Touchdowns"].std()
    merged["clutch_z"] = (merged["Clutch Score"] - merged["Clutch Score"].mean()) / merged["Clutch Score"].std()

    merged["score_numerator"] = (
        0.15*merged["pyd_z"] + 0.15*merged["comp_z"] + 0.25*merged["ptd_z"] 
        - 0.20*merged["int_z"] + 0.05*merged["rtd_z"] + 0.05*merged["ryd_z"] + 0.15*merged["clutch_z"]
    )

    merged["w_salary"] = 1 + 0.0000001 * merged["Salary"]
    merged["QPI"] = (merged["score_numerator"] / merged["w_salary"]).round(4)

    fig = px.bar(
        merged, x="QPI", y="Player", 
        title="QB Performance Index", color="QPI", color_continuous_scale="Viridis"
    )
    fig.show()

insight_3()


### Visualization 3 Explanation

This visualization displays the QPI for each quarterback in the 2024 NFL season in a horizontal bar chart. As discussed earlier, QPI is calculated through a weighted, normalized sum of the individual statistics and clutch score divided by a factor of the salary. This bar chart allows NFL teams to easily see overall which quarterbacks perform better. Quarterbacks with a positive QPI ("pointing right") are considered undervalued/to perform better and quarterbacks with a negative QPI ("pointing left") are considered overvalued/to perform worse. An NFL GM would find such a graph very useful.

# Cited Sources

If you used any additional sources to complete your Data Analysis section, list them here:


*   Example Module Documentation
*   Example Stack Overflow Assistance


https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html

https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html



# Graphical User Interface (GUI) Implementation
If you decide to create a GUI for Phase II, please create a separate Python file (.py) to build your GUI. You must submit both the completed PhaseII.ipynb and your Python GUI file.

# Submission

Prior to submitting your notebook to Gradescope, be sure to <b>run all functions within this file</b>. We will not run your functions ourselves, so we must see your outputs within this file in order to receive full credit.
